In [1]:
import sqlite3
import pandas as pd

df = pd.read_csv("../data/raw/2024/bikeshare-ridership-2024.csv")

In [2]:
df.head()

,Trip_Id,Trip_Duration,Start_Station_Id,Start_Time,Start_Station_Name,End_Station_Id,End_Time,End_Station_Name,Bike_Id,User_Type,Bike_Model
0,26682740,528,7041,2024-01-01 00:00:08,Edward St / Yonge St,7130.0,2024-01-01 00:08:56,Davenport Rd / Bedford Rd,7855,Casual,EFIT G5
1,26682742,1924,7245,2024-01-01 00:02:04,Tecumseth St / Queen St W - SMART,7259.0,2024-01-01 00:34:08,Lower Spadina Ave / Lake Shore Blvd W,2453,Casual,ICONIC
2,26682743,2741,7052,2024-01-01 00:03:25,Wellington St W / Bay St,7366.0,2024-01-01 00:49:06,Fort York Blvd / Bathurst St - SMART,7332,Member,ICONIC
3,26682744,353,7041,2024-01-01 00:03:39,Edward St / Yonge St,7418.0,2024-01-01 00:09:32,College Park - Yonge St Entrance,782,Member,ICONIC
4,26682746,2118,7721,2024-01-01 00:04:44,Widmer St / King St W,7721.0,2024-01-01 00:40:02,Widmer St / King St W,1008,Member,ICONIC


In [3]:
set(df["Bike_Model"])

{'EFIT', 'EFIT G5', 'ICONIC'}

In [4]:
df["is_ebike"] = df["Bike_Model"].isin(["EFIT","EFIT"])

In [5]:
df.columns

Index(['Trip_Id', 'Trip_Duration', 'Start_Station_Id', 'Start_Time',
       'Start_Station_Name', 'End_Station_Id', 'End_Time', 'End_Station_Name',
       'Bike_Id', 'User_Type', 'Bike_Model', 'is_ebike'],
      dtype='str')

In [22]:
df.isna().sum()

Trip_Id                  0
Trip_Duration            0
Start_Station_Id         0
Start_Time               0
Start_Station_Name       0
End_Station_Id        3289
End_Time                 0
End_Station_Name      3289
Bike_Id                  0
User_Type                0
Bike_Model               0
is_ebike                 0
dtype: int64

In [7]:
df = df[df["Trip_Duration"] >= 60]

In [8]:
df["Trip_Duration"].quantile([0.99,0.999,0.9999])

0.9900     4895.0000
0.9990    14870.7560
0.9999    99216.2346
Name: Trip_Duration, dtype: float64

In [9]:
df = df[df["Trip_Duration"] <= 12875.00]

In [15]:
df.dtypes

Trip_Id                        int64
Trip_Duration                  int64
Start_Station_Id               int64
Start_Time            datetime64[us]
Start_Station_Name               str
End_Station_Id                 Int64
End_Time              datetime64[us]
End_Station_Name                 str
Bike_Id                        int64
User_Type                        str
Bike_Model                       str
is_ebike                        bool
dtype: object

In [12]:
df["Start_Time"] = pd.to_datetime(df["Start_Time"])
df["End_Time"] = pd.to_datetime(df["End_Time"])

In [14]:
df["End_Station_Id"] = df["End_Station_Id"].astype("Int64")

In [16]:
(df["Trip_Id"].duplicated()).sum()

np.int64(0)

In [17]:
df["User_Type"].value_counts()

User_Type
Member    5351161
Casual    1575163
Name: count, dtype: int64

In [18]:
(df["Start_Station_Id"] == df["End_Station_Id"]).sum()

np.int64(184851)

In [19]:
round_trips = df[df["Start_Station_Id"] == df["End_Station_Id"]]

In [20]:
round_trips["Trip_Duration"].describe()

count    184851.000000
mean       1985.887363
std        2077.288457
min          61.000000
25%         477.000000
50%        1333.000000
75%        2744.000000
max       12874.000000
Name: Trip_Duration, dtype: float64

In [21]:
is_redock =(df["Start_Station_Id"] == df["End_Station_Id"]) & (df["Trip_Duration"] < 120)
df = df[~is_redock]

In [1]:
conn = sqlite3.connect("../data/processed/bikeshare.db")
df.to_sql("trips", conn, if_exists="append", index=False)
conn.close()

NameError: name 'sqlite3' is not defined